In [ ]:
# The transformers library is still necessary for the underlying BART model architecture.
from transformers import BartTokenizer, BartConfig, BartForConditionalGeneration

# Define the model name from the Hugging Face Hub.
# This tells the library where to download the necessary files from.
model_name = "Babelscape/rebel-large"

# --- Step 1: Manually load the Tokenizer, Configuration, and Model ---

# Load the specific BART tokenizer from the pre-trained model files.
print("Loading tokenizer...")
tokenizer = BartTokenizer.from_pretrained(model_name)

# Load the specific BART model configuration.
# This file contains all the model's architectural settings.
print("Loading model configuration...")
config = BartConfig.from_pretrained(model_name)

# Load the BART model for sequence-to-sequence tasks with the pre-trained weights.
print("Loading pre-trained model weights...")
model = BartForConditionalGeneration.from_pretrained(model_name, config=config)
print("Model loaded successfully.")

In [ ]:
text = '''The Battle of Sinhagad, also known as Battle of Kondhana, involved an attack by Marathas during the night of 4 February 1670 on the Mughal fort of Sinhagad (then Kondhana), near the city of Pune, Maharashtra. The Marathas captured the fort. Battle Sinhagad was one of the first forts which Shivaji recaptured from the Mughals following the treaty of Purandar. The capture was made possible by scaling the walls at night with rope ladders. Tanaji Malusare was killed, but the fort was captured by the Marathas. The battle and Tanaji's exploits are the basis of a popular Marathi ballad.[2] Aftermath When Shivaji was informed about the victory and the loss of Tanaji's life during the battle, he is said to have exclaimed "गड आला पण सिंह गेला" (IAST: Gaḍa ālā paṇa siṃha gela English: The fort has been captured but we lost the lion). A bust of Tanaji Malusare was installed at the fort in memory of his contribution and sacrifice.[3] In popular culture Tanhaji - a Hindi movie of 2020, starring Ajay Devgan, depicted the battle of Kondhana. Raja Shivchatrapati - a Marathi TV serial of Star Pravah had episodes about this battle. Subhedar - a Marathi movie of 2023, starring Chinmay Mandlekar, depicted the battle of Kondhana.'''

# Tokenize the text
input_ids = tokenizer(text, return_tensors="pt").input_ids

# Generate the linearized triplets
generated_ids = model.generate(input_ids)

# Decode the output
decoded_output = tokenizer.decode(generated_ids[0], skip_special_tokens=False)

# The output will be in REBEL's linearized format
# e.g., '<triplet> Company <subj> Employee <obj> party to agreement'
print(decoded_output)

In [ ]:
import csv

# Assume 'decoded_output' contains the raw string from the REBEL model.
# --- 1. Parsing Logic (from previous script) ---
# Clean up the start and end tokens from the raw output.
decoded_output = decoded_output.replace("<s>", "").replace("</s>", "")

triplets = []
# Split the string by the '<triplet>' token to process each group.
relation_groups = decoded_output.split('<triplet>')

current_head_entity = None
for group in relation_groups:
    if not group.strip():
        continue
        
    # Split the group into the head entity and the rest of the relations.
    parts = group.split('<subj>')
    head_entity = parts[0].strip()
    
    # Process each relation associated with the head entity.
    for relation_part in parts[1:]:
        try:
            tail_entity, relation = relation_part.split('<obj>')
            # Append the extracted triplet as a tuple.
            triplets.append((head_entity.strip(), relation.strip(), tail_entity.strip()))
        except ValueError:
            # Skip any malformed relation parts.
            continue

# --- 2. Save the Triplets to a CSV file ---
csv_file_name = "triplets.csv"

# 'with open' ensures the file is properly closed after writing.
with open(csv_file_name, mode='w', newline='', encoding='utf-8') as file:
    # Create a csv writer object.
    writer = csv.writer(file)
    
    # Write the header row.
    writer.writerow(['head', 'relation', 'tail'])
    
    # Write all the triplet data rows.
    writer.writerows(triplets)

print(f"✅ Successfully saved {len(triplets)} triplets to '{csv_file_name}'")